In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.getActiveSession()

if spark is None:
    spark = SparkSession.builder.appName("Databricks ML Pipeline").master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/30 00:26:31 WARN Utils: Your hostname, Richards-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.181 instead (on interface en0)
26/06/30 00:26:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/30 00:26:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
wnba_raw = spark.read.csv("data/wnba_raw.csv", header=True, inferSchema=True)

In [4]:
#Getting all of the game_id values from the raw dataframe
game_ids = [row.game_id for row in wnba_raw.select("game_id").distinct().collect() if row.game_id is not None and row.game_id.isnumeric()]
print(game_ids)


['401857000', '401856991', '401856984', '401856998', '401857005', '401856988', '401856993', '401856996', '401856995', '401857001', '401857003', '401856989', '401856987', '401857008', '401856999', '401856992', '401857004', '401856997', '401856986', '401856994', '401856985', '401857007', '401857006', '401856990', '401857002', '401856968', '401856965', '401856973', '401856979', '401856963', '401856975', '401856977', '401856962', '401856966', '401856971', '401856978', '401856969', '401856981', '401856976', '401856980', '401856972', '401856982', '401856974', '401856983', '401856967', '401856961', '401856970', '401856964', '401856952', '401856949', '401856944', '401856942', '401856948', '401856945', '401856946', '401856956', '401856954', '401856955', '401856957', '401856960', '401856937', '401856947', '401856953', '401856938', '401856950', '401856940', '401856959', '401856958', '401856936', '401856943', '401856941', '401856939', '401856951', '401856925', '401856914', '401856926', '401856934'

In [5]:
#Creating separate dataframes for each game_id and storing them in a dictionary
game_by_game_dfs = {game_id: wnba_raw.filter(wnba_raw.game_id == game_id) for game_id in game_ids}
game_by_game_dfs['401856986'].show(10)

26/06/30 00:26:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------+-----------+---------------+-------+--------------------+--------------------+----------+----------+-------------+--------------------+-------------------+------------+-----------+-------+------------+------------+------------+--------------------+-------------+----------------+----------------+----------------+-----------------+---------+------+-----------+------------+--------------+----------------+----------------+------------------+------------+--------------+----------------+----------------+------------------+-----------+-------------+---------------------+----------------+---+-----+-------------+-------------+-------------------+-------------------+----+---------+--------+---------+-------------------------------+----------------------------+----------------------------+-----------------------------+--------------------------+--------------------------+------+-------+--------+---------------+------------+----------+-------------------+-----------------+
|game_

In [15]:
teams = spark.read.csv("data/wnba_teams.csv", header = True)
teams = teams.drop("fox_section")
teams = teams.withColumnRenamed("fox_team_id", "team_id")
teams = teams.withColumnRenamed("fox_team_name", "team_name")
teams.head(10)

[Row(team_id='1', team_name='Atlanta Dream'),
 Row(team_id='5', team_name='New York Liberty'),
 Row(team_id='4', team_name='Indiana Fever'),
 Row(team_id='6', team_name='Washington Mystics'),
 Row(team_id='29', team_name='Toronto Tempo'),
 Row(team_id='2', team_name='Chicago Sky'),
 Row(team_id='3', team_name='Connecticut Sun'),
 Row(team_id='9', team_name='Minnesota Lynx'),
 Row(team_id='11', team_name='Las Vegas Aces'),
 Row(team_id='23', team_name='Golden State Valkyries')]

Now I'm going to create a player dataframe that matches each player to their team based on the team ID's

In [16]:

raw_rosters = spark.read.csv("data/rosters/*", header = True)
complete_rosters = raw_rosters.join(teams, on = "team_id", how = "left")
complete_rosters.head(10)


26/06/30 00:33:04 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/rosters/*.
java.io.FileNotFoundException: File data/rosters/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource

[Row(team_id='23', position_group='GUARD', player='Veronica Burton', pos='G', age='25', ht='5\'9""', wt='155 lbs', college='-', athlete_id='641', team_name='Golden State Valkyries'),
 Row(team_id='23', position_group='GUARD', player='Kaila Charles', pos='G/F', age='28', ht='6\'1""', wt='168 lbs', college='-', athlete_id='557', team_name='Golden State Valkyries'),
 Row(team_id='23', position_group='GUARD', player='Kaitlyn Chen', pos='G', age='24', ht='5\'9""', wt='-', college='-', athlete_id='873', team_name='Golden State Valkyries'),
 Row(team_id='23', position_group='GUARD', player='Tiffany Hayes', pos='G', age='36', ht='5\'10""', wt='155 lbs', college='Connecticut', athlete_id='30', team_name='Golden State Valkyries'),
 Row(team_id='23', position_group='GUARD', player='Juste Jocyte', pos='G/F', age='20', ht='6\'0""', wt='-', college='-', athlete_id='850', team_name='Golden State Valkyries'),
 Row(team_id='23', position_group='GUARD', player='Miela Sowah', pos='G', age='26', ht='5\'10